In [1]:
import math
from tqdm import tqdm
import nltk
from nltk.corpus import stopwords
import ssl
import os
from pathlib import Path
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
import pandas as pd
from datetime import datetime
import pickle
import re
import pyxdameraulevenshtein
# import apsw  # Commented out - not needed for CSV processing
import sys
import numpy as np
import corp_simplify_utils
import seaborn as sns
import matplotlib.pyplot as plt
import pyreadr
from collections import Counter

# nlp
import spacy
from spacy import displacy
# Remove problematic direct import: import en_core_web_lg
# Use spacy.load() instead (loaded below when needed)

# analysis/regressions
import statsmodels.api as sm
from statsmodels.formula.api import glm
from statsmodels.genmod.families import Poisson
from scipy.stats import ks_2samp
from scipy.stats import mannwhitneyu
from scipy.stats import ttest_ind

# from statsmodels.graphics.gofplots import qqplot_2samples
from scipy import stats
from joypy import joyplot
from matplotlib import cm

from datetime import date
today_for_filenames = date.today()
curr_date = str(today_for_filenames.strftime("%Y%m%d"))


NUMBER_OF_MATCHES_TO_RECORD = 10
punc_remove_re = re.compile(r'\W+')
corp_re = re.compile('( (group|holding(s)?( co)?|inc(orporated)?|ltd|l ?l? ?[cp]|co(rp(oration)?|mpany)?|s[ae]|plc))+$')
and_re = re.compile(' & ')
punc1_re = re.compile(r"(?<=\S)['\u00B4\.](?=\S)")  # Fixed unicode character
punc2_re = re.compile(r"[\s\.,:;/'\"`\u00B4\u2018\u2019\u201C\u201D\(\)\[\]\{\}_\u2014\-?$=!]+")  # Fixed unicode characters

STOPWORDS = nltk.corpus.stopwords.words('english')
STOPWORDS.remove("am")
STOPWORDS.remove("up")
STOPWORDS.remove("in")
STOPWORDS.remove("on")
STOPWORDS.remove("all")
STOPWORDS.remove("any")
STOPWORDS.remove("most")
STOPWORDS.remove("no")
STOPWORDS.remove("nor")
STOPWORDS.remove("own")
STOPWORDS.remove("same")
STOPWORDS.remove("so")
STOPWORDS.remove("very")
STOPWORDS.remove("s")
STOPWORDS.remove("t")
STOPWORDS.remove("d")
STOPWORDS.remove("ll")
STOPWORDS.remove("m")
STOPWORDS.remove("o")
STOPWORDS.remove("re")
STOPWORDS.remove("ve")
STOPWORDS.remove("y")

#compile regex patterns to reuse
STOPWORD_RE = re.compile(r'\b(the|of|and|in|on)\b', re.IGNORECASE)
CORP_SUFFIX_RE = re.compile(r'\b(inc|corp|ltd|llc|plc|co|company|limited)\b', re.IGNORECASE)
PDF_PATTERN_RE = re.compile(r'\s[0-9]*\s[km]b\s*pdf', re.IGNORECASE)
PUNCT_RE = re.compile(r'[^\w\s-]')  # match punctuation
MULTISPACE_RE = re.compile(r'\s+')

stopword_re_str = r""
for word in STOPWORDS:
	stopword_re_str += r'\b' + word + r'\b|'
stopword_re = re.compile(stopword_re_str[:-1]) # The negative 1 is for the fencepost |

# Commented out - these paths don't exist in your project
# BASE_DIR = "/Users/aawesomez/Documents/UROP/NLP-regextable/"
# DB_PATH = BASE_DIR + "Data/master.sqlite"
# LAST_SAVE_DATASET_DATE = "20220402"

# Function to calculate longest common substring, from https://www.geeksforgeeks.org/print-longest-common-substring/
# function to find and print 
# the longest common substring of
# X[0..m-1] and Y[0..n-1]
def get_longest_common_substring(X, Y, m, n):
 
    # Create a table to store lengths of
    # longest common suffixes of substrings.
    # Note that LCSuff[i][j] contains length
    # of longest common suffix of X[0..i-1] and
    # Y[0..j-1]. The first row and first
    # column entries have no logical meaning,
    # they are used only for simplicity of program
    LCSuff = [[0 for i in range(n + 1)]
                 for j in range(m + 1)]
 
    # To store length of the
    # longest common substring
    length = 0
 
    # To store the index of the cell
    # which contains the maximum value.
    # This cell's index helps in building
    # up the longest common substring
    # from right to left.
    row, col = 0, 0
 
    # Following steps build LCSuff[m+1][n+1]
    # in bottom up fashion.
    for i in range(m + 1):
        for j in range(n + 1):
            if i == 0 or j == 0:
                LCSuff[i][j] = 0
            elif X[i - 1] == Y[j - 1]:
                LCSuff[i][j] = LCSuff[i - 1][j - 1] + 1
                if length < LCSuff[i][j]:
                    length = LCSuff[i][j]
                    row = i
                    col = j
            else:
                LCSuff[i][j] = 0
 
    # if true, then no common substring exists
    if length == 0:
        return ""
 
    # allocate space for the longest
    # common substring
    resultStr = ['0'] * length
 
    # traverse up diagonally form the
    # (row, col) cell until LCSuff[row][col] != 0
    while LCSuff[row][col] != 0:
        length -= 1
        resultStr[length] = X[row - 1] # or Y[col-1]
 
        # move diagonally up to previous cell
        row -= 1
        col -= 1
 
    # required longest common substring
    longest_common_substring = ''.join(resultStr)

    return longest_common_substring


# Function from Brad Hackinen's NAMA
def basicHash(s):
    '''
    A simple case and puctuation-insensitive hash
    '''
    s = s.lower()
    s = re.sub(and_re,' and ',s)
    s = re.sub(punc1_re,'',s)
    s = re.sub(punc2_re,' ',s)
    s = s.strip()

    return s

# Function from Brad Hackinen's NAMA
def corpHash(s):
    '''
    A hash function for corporate subsidiaries
    Insensitive to
        -case & punctation
        -'the' prefix
        -common corporation suffixes, including 'holding co'
    '''
    s = basicHash(s)
    if s.startswith('the '):
        s = s[4:]

    s = re.sub(corp_re,'',s,count=1)

    return s

# function to clean org names
def clean_fin_org_names(name: str) -> str:
    if name is None or not isinstance(name, str) or name == "NA":
        return ""
    
    # James strip metadata from name
    # name = name.split(',')[0]
    #Remove patterns like "10 kb pdf"
    name = PDF_PATTERN_RE.sub("", name)

    #Unicode and punctuation cleanup
    name = corp_simplify_utils.normalize_unicode(name)
    name = PUNCT_RE.sub(" ", name)

    #Remove corporate suffixes and stopwords
    name = CORP_SUFFIX_RE.sub("", name)
    name = STOPWORD_RE.sub("", name)

    #Normalize spacing and lowercase
    name = MULTISPACE_RE.sub(" ", name).strip().lower()

    return name


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/stevenkang/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# Locate data directory and read in data files

current_dir = Path(os.getcwd()).parent
data_dir = current_dir / 'data'
data_dir = data_dir.resolve()

try:
    compustat_df = pd.read_csv(data_dir / 'CompustatNames.csv')
    cik_df = pd.read_csv(data_dir / 'CIK.csv')
    fdic_df = pd.read_csv(data_dir / 'FDIC_clean.csv') # Using your 'FDIC_clean.csv'
    sec_df = pd.read_csv(data_dir / 'SEC_Institutions.csv')
except FileNotFoundError as e:
    print(f"Error loading file: {e}")
    print(f"Please make sure your CSV files are in the directory: {data_dir}")
    exit()
    

In [3]:
# cleaning and standardizing organization names
compustat_df['std_name'] = compustat_df['conm'].apply(clean_fin_org_names)
fdic_df['std_name'] = fdic_df['NAME'].apply(clean_fin_org_names)
sec_df['std_name'] = sec_df['Name'].apply(clean_fin_org_names)
cik_df['std_name'] = cik_df['company_name'].apply(clean_fin_org_names)

In [4]:
cik_df.head(10)

,Unnamed: 0,company_name,cik,std_name
0,0,!J INC,1438823.0,j
1,1,"#1 A LIFESAFER HOLDINGS, INC.",1509607.0,1 a lifesafer holdings
2,2,#1 ARIZONA DISCOUNT PROPERTIES LLC,1457512.0,1 arizona discount properties
3,3,#1 PAINTBALL CORP,1433777.0,1 paintball
4,4,$ LLC,1427189.0,
5,5,"$AVY, INC.",1655250.0,avy
6,6,& S MEDIA GROUP LLC,1447162.0,s media group
7,7,&TV COMMUNICATIONS INC.,1479357.0,tv communications
8,8,"&VEST DOMESTIC FUND II KPIV, L.P.",1802417.0,vest domestic fund ii kpiv l p
9,9,&VEST DOMESTIC FUND II LP,1800903.0,vest domestic fund ii lp


In [5]:
sec_df.head(10)

,index,CIK,Ticker,Name,Exchange,SIC,Business,Incorporated,IRS,std_name
0,0,1090872,A,Agilent Technologies Inc,NYSE,3825.0,CA,DE,770518772.0,agilent technologies
1,1,4281,AA,Alcoa Inc,NYSE,3350.0,PA,PA,250317820.0,alcoa
2,2,1332552,AAACU,Asia Automotive Acquisition Corp,NaN,6770.0,DE,DE,203022522.0,asia automotive acquisition
3,3,1287145,AABB,Asia Broadband Inc,OTC,8200.0,GA,NV,721569126.0,asia broadband
4,4,1024015,AABC,Access Anytime Bancorp Inc,NaN,6035.0,NM,DE,850444597.0,access anytime bancorp
5,5,1099290,AAC,Sinocoking Coal & Coke Chemical Industries Inc,NASDAQ,3312.0,F4,FL,593404233.0,sinocoking coal coke chemical industries
6,6,1264707,AACC,Asset Acceptance Capital Corp,NaN,6153.0,MI,NaN,800076779.0,asset acceptance capital
7,7,849116,AACE,Ace Cash Express Inc,NaN,6099.0,TX,TX,752142963.0,ace cash express
8,8,1409430,AAGC,All American Gold Corp,OTC,1040.0,IN,WY,260665571.0,all american gold
9,9,948846,AAI,Airtran Holdings Inc,NaN,4512.0,FL,NV,582189551.0,airtran holdings


In [6]:
compustat_df.head(10)

,Unnamed: 0,gvkey,conm,tic,cusip,cik,sic,naics,gsubind,gind,year1,year2,std_name
0,1,1004,AAR CORP,AIR,000361105,1750.0,5080.0,423860.0,20101010.0,201010.0,1965,2020,aar
1,2,1013,ADC TELECOMMUNICATIONS INC,ADCT.1,000886309,61478.0,3661.0,334210.0,45201020.0,452010.0,1974,2010,adc telecommunications
2,3,1045,AMERICAN AIRLINES GROUP INC,AAL,02376R102,6201.0,4512.0,481111.0,20302010.0,203020.0,1950,2021,american airlines group
3,4,1050,CECO ENVIRONMENTAL CORP,CECE,125141101,3197.0,3564.0,333413.0,20201050.0,202010.0,1974,2021,ceco environmental
4,5,1062,ASA GOLD AND PRECIOUS METALS,ASA,G3156P103,1230869.0,6799.0,523999.0,40203010.0,402030.0,1966,2021,asa gold precious metals
5,6,1072,AVX CORP,AVX,002444107,859163.0,3670.0,334416.0,45203015.0,452030.0,1973,2018,avx
6,7,1075,PINNACLE WEST CAPITAL CORP,PNW,723484101,764622.0,4911.0,2211.0,55101010.0,551010.0,1951,2021,pinnacle west capital
7,8,1076,PROG HOLDINGS INC,PRG,74319R101,1808834.0,6141.0,522220.0,40202010.0,402020.0,1981,2021,prog holdings
8,9,1078,ABBOTT LABORATORIES,ABT,002824100,1800.0,3845.0,334510.0,35101010.0,351010.0,1950,2021,abbott laboratories
9,10,1082,SERVIDYNE INC,SERV.1,81765M106,1923.0,8700.0,541310.0,20201050.0,202010.0,1977,2010,servidyne


In [7]:
fdic_df.head(10)

,NAME,NAMEHCR,STALP,STNAME,BKCLASS,ASSET,CERT,FED_RSSD,org_name,commented,Commented,mean_ASSET,median_ASSET,mean_ASSET_type,median_ASSET_type,std_name
0,The Southington Bank and Trust Company,NaN,CT,Connecticut,NM,4.857000e+07,4,573401,the southington bank and trust company,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000,southington bank trust
1,Colonial Bank of Waterbury,NaN,CT,Connecticut,NM,6.246550e+08,6,148304,colonial bank of waterbury,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000,colonial bank waterbury
2,Fleet Bank of Maine,NaN,ME,Maine,SM,1.699404e+09,8,422406,fleet bank of maine,False,Did not comment,9.155463e+08,95742500,9.804661e+08,131926000,fleet bank maine
3,Union Trust Company,NaN,ME,Maine,SM,5.391690e+08,9,563907,union trust company,False,Did not comment,9.155463e+08,95742500,9.804661e+08,131926000,union trust
4,Northeast Bank of Sanford,NaN,ME,Maine,SM,5.569200e+07,10,112109,northeast bank of sanford,False,Did not comment,9.155463e+08,95742500,9.804661e+08,131926000,northeast bank sanford
5,State Street Bank and Trust Company,State Street Corporation,MA,Massachusetts,SM,2.335429e+11,14,35301,state street bank and trust company,True,Commented,2.565979e+09,145717500,3.941421e+09,218865500,state street bank trust
6,BayBank Harvard Trust Company,NaN,MA,Massachusetts,NM,1.435310e+09,18,852209,baybank harvard trust company,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000,baybank harvard trust
7,Durfee Attleboro Bank,NaN,MA,Massachusetts,NM,3.468080e+08,20,202402,durfee attleboro bank,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000,durfee attleboro bank
8,Bank of New England - North Shore,NaN,MA,Massachusetts,SM,1.532740e+08,21,658205,bank of new england - north shore,False,Did not comment,9.155463e+08,95742500,9.804661e+08,131926000,bank new england - north shore
9,Shawmut Bank of Franklin County,NaN,MA,Massachusetts,NM,1.597270e+08,22,661205,shawmut bank of franklin county,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000,shawmut bank franklin county


In [8]:
# Test if CIK is already in SEC
sec_ciks = set(sec_df['CIK'].dropna())
cik_ciks = set(cik_df['cik'].dropna())

print(f"CIKs in sec_df: {len(sec_ciks)}")
print(f"CIKs in cik_df: {len(cik_ciks)}")
print(f"Is SEC_Institutions.csv a subset of CIK.csv? {sec_ciks.issubset(cik_ciks)}")

compustat_ciks = set(compustat_df['cik'].dropna())
print(f"CIKs in compustat_df: {len(compustat_ciks)}")
print(f"Is CompustatNames.csv a subset of CIK.csv? {compustat_ciks.issubset(cik_ciks)}")


CIKs in sec_df: 13737
CIKs in cik_df: 806225
Is SEC_Institutions.csv a subset of CIK.csv? True
CIKs in compustat_df: 12835
Is CompustatNames.csv a subset of CIK.csv? False


SEC_Institutions.csv is a subset of CIK.csv --- > Don't need to merge SEC into the crosswalk. 

In [9]:
# renaming columns for consistency
compustat_temp = compustat_df[['std_name', 'conm', 'tic', 'cusip', 'cik']].rename(columns={'conm': 'raw_name'})
compustat_temp['source'] = 'compustat'

fdic_temp = fdic_df[['std_name', 'NAME']].rename(columns={'NAME': 'raw_name'})
fdic_temp['source'] = 'fdic'

cik_temp = cik_df[['std_name', 'company_name', 'cik']].rename(columns={'company_name': 'raw_name'})
cik_temp['source'] = 'cik'

# Combine all into one long dataframe
all_names_df = pd.concat([compustat_temp, fdic_temp, cik_temp], ignore_index=True)

# Drop any rows where cleaning failed (no std_name)
all_names_df = all_names_df.dropna(subset=['std_name'])
# Remove any empty std_name entries
all_names_df = all_names_df[all_names_df['std_name'] != ""]

# Cleaning up CIKs and FED_RSSD to be strings without decimal points
for col in ['CIK', 'FED_RSSD']:
    if col in all_names_df.columns:
        # Convert to string after converting to int to remove any decimal points
        all_names_df[col] = all_names_df[col].dropna().astype(float).astype(int).astype(str)

print(f"Total entries to match: {len(all_names_df)}")
all_names_df.head(20)

Total entries to match: 915294


,std_name,raw_name,tic,cusip,cik,source
0,aar,AAR CORP,AIR,000361105,1750.0,compustat
1,adc telecommunications,ADC TELECOMMUNICATIONS INC,ADCT.1,000886309,61478.0,compustat
2,american airlines group,AMERICAN AIRLINES GROUP INC,AAL,02376R102,6201.0,compustat
3,ceco environmental,CECO ENVIRONMENTAL CORP,CECE,125141101,3197.0,compustat
4,asa gold precious metals,ASA GOLD AND PRECIOUS METALS,ASA,G3156P103,1230869.0,compustat
5,avx,AVX CORP,AVX,002444107,859163.0,compustat
6,pinnacle west capital,PINNACLE WEST CAPITAL CORP,PNW,723484101,764622.0,compustat
7,prog holdings,PROG HOLDINGS INC,PRG,74319R101,1808834.0,compustat
8,abbott laboratories,ABBOTT LABORATORIES,ABT,002824100,1800.0,compustat
9,servidyne,SERVIDYNE INC,SERV.1,81765M106,1923.0,compustat


In [10]:
grouped_by_cik_id = all_names_df.groupby('cik')
confident_matches = []

for cik_value, group in grouped_by_cik_id:
    if len(group) > 1:
        # Aggregate the data based on cik
        keys = {
            'cik': group['cik'].dropna().unique().tolist(),
            # Now aggregate the std_name to see all variations found for cik
            'standardized_names': '|'.join(group['std_name'].dropna().unique()),
            # Aggregate other fields as before
            'aliases': '|'.join(group['raw_name'].dropna().unique()),
            'sources': ','.join(group['source'].unique())
        }
        confident_matches.append(keys)
        
pd.set_option('display.max_colwidth', None)
confident_crosswalk_id_based = pd.DataFrame(confident_matches)
print(f"Found {len(confident_crosswalk_id_based)} confident entity clusters.")

Found 56760 confident entity clusters.


In [11]:
confident_crosswalk_id_based.head(10)

,cik,standardized_names,aliases,sources
0,[1750.0],aar,AAR CORP,"compustat,cik"
1,[1800.0],abbott laboratories,ABBOTT LABORATORIES,"compustat,cik"
2,[1841.0],abel noser bd|abel noser,ABEL NOSER CORP /BD|ABEL/NOSER CORP.,cik
3,[1853.0],aberdeen idaho mining|motivnation,"ABERDEEN IDAHO MINING CO|MOTIVNATION, INC.",cik
4,[1860.0],thomson richard william bd|thomson richard william,"THOMSON RICHARD WILLIAM /BD|THOMSON, RICHARD WILLIAM",cik
5,[1904.0],abraham bd|abraham|abraham securities corporation,"ABRAHAM & CO INC /BD|ABRAHAM & CO., INC.|ABRAHAM SECURITIES CORPORATION",cik
6,[1918.0],abrams allan edward|homeland securities financial services group|merchanthouse securities|wizer financial,"ABRAMS, ALLAN EDWARD|HOMELAND SECURITIES FINANCIAL SERVICES GROUP, INC.|THE MERCHANTHOUSE SECURITIES, INC.|WIZER FINANCIAL. INC.",cik
7,[1923.0],servidyne|abrams industries,"SERVIDYNE INC|ABRAMS INDUSTRIES INC|SERVIDYNE, INC.","compustat,cik"
8,[1961.0],worlds|academic computer systems|worlds com,"WORLDS INC|ACADEMIC COMPUTER SYSTEMS INC|WORLDS COM INC|WORLDS.COM, INC.","compustat,cik"
9,[2034.0],aceto,ACETO CORP,"compustat,cik"


Able to create a table of 56760 companies based on CIK id.

In [12]:
# Find all the rows where there is duplicate CIK in one of the dataframes
# In other words, there are multiple aliases, but coming from the same source

confident_crosswalk_id_based[
    (confident_crosswalk_id_based['aliases'].str.contains('\|')) & 
    (~confident_crosswalk_id_based['sources'].str.contains(r','))
].head(20)

,cik,standardized_names,aliases,sources
2,[1841.0],abel noser bd|abel noser,ABEL NOSER CORP /BD|ABEL/NOSER CORP.,cik
3,[1853.0],aberdeen idaho mining|motivnation,"ABERDEEN IDAHO MINING CO|MOTIVNATION, INC.",cik
4,[1860.0],thomson richard william bd|thomson richard william,"THOMSON RICHARD WILLIAM /BD|THOMSON, RICHARD WILLIAM",cik
5,[1904.0],abraham bd|abraham|abraham securities corporation,"ABRAHAM & CO INC /BD|ABRAHAM & CO., INC.|ABRAHAM SECURITIES CORPORATION",cik
6,[1918.0],abrams allan edward|homeland securities financial services group|merchanthouse securities|wizer financial,"ABRAMS, ALLAN EDWARD|HOMELAND SECURITIES FINANCIAL SERVICES GROUP, INC.|THE MERCHANTHOUSE SECURITIES, INC.|WIZER FINANCIAL. INC.",cik
11,[2093.0],acme metals de|acme metals,ACME METALS INC /DE/|ACME METALS INC/,cik
13,[2110.0],acorn investment trust|columbia acorn trust|liberty acorn trust,ACORN INVESTMENT TRUST|COLUMBIA ACORN TRUST|LIBERTY ACORN TRUST,cik
17,[2310.0],am international|multigraphics,AM INTERNATIONAL INC|MULTIGRAPHICS INC,cik
18,[2380.0],administrative data management ta|foresters investor services ta,ADMINISTRATIVE DATA MANAGEMENT CORP /TA|ADMINISTRATIVE DATA MANAGEMENT CORP /TA|FORESTERS INVESTOR SERVICES INC/TA,cik
21,[2554.0],aei securities bd|aei securities,"AEI SECURITIES INC /BD|AEI SECURITIES, INC.",cik


There appears to be many aliases for the same CIK ID in the cik.csv file

In [21]:
# Create a copy of the confident_crosswalk_id_based DataFrame
crosswalk_with_fdic = confident_crosswalk_id_based.copy()

# Group the FDIC data by the standardized name, joining aliases with '|'
fdic_grouped = fdic_df.groupby('std_name')['NAME'].agg('|'.join).reset_index()
fdic_grouped = fdic_grouped.rename(columns={'NAME': 'all_fdic_aliases'})

crosswalk_with_fdic = crosswalk_with_fdic.merge(
    fdic_grouped,
    how='left',
    left_on='standardized_names',
    right_on='std_name'
)

# Get filter for rows with non-NA FDIC aliases
not_na_filter = crosswalk_with_fdic['all_fdic_aliases'].notna()
aliases_series = crosswalk_with_fdic.loc[not_na_filter, 'aliases'].fillna('').astype(str)
# Get the new FDIC aliases (already pipe-separated strings)
fdic_aliases_series = crosswalk_with_fdic.loc[not_na_filter, 'all_fdic_aliases']
has_existing_aliases_mask = (aliases_series != '')
# Get the index locations for rows to update
idx_with_existing = aliases_series[has_existing_aliases_mask].index

# Join aliases appropriately
idx_without_existing = aliases_series[~has_existing_aliases_mask].index
crosswalk_with_fdic.loc[idx_with_existing, 'aliases'] = \
    aliases_series.loc[idx_with_existing] + '|' + fdic_aliases_series.loc[idx_with_existing]
crosswalk_with_fdic.loc[idx_without_existing, 'aliases'] = \
    fdic_aliases_series.loc[idx_without_existing]
    
# Add 'FDIC' to sources
crosswalk_with_fdic.loc[not_na_filter, 'sources'] = \
    crosswalk_with_fdic['sources'].fillna('') + ',FDIC'

crosswalk_with_fdic.head(10)

,cik,standardized_names,aliases,sources,std_name,all_fdic_aliases
0,[1750.0],aar,AAR CORP,"compustat,cik",NaN,NaN
1,[1800.0],abbott laboratories,ABBOTT LABORATORIES,"compustat,cik",NaN,NaN
2,[1841.0],abel noser bd|abel noser,ABEL NOSER CORP /BD|ABEL/NOSER CORP.,cik,NaN,NaN
3,[1853.0],aberdeen idaho mining|motivnation,"ABERDEEN IDAHO MINING CO|MOTIVNATION, INC.",cik,NaN,NaN
4,[1860.0],thomson richard william bd|thomson richard william,"THOMSON RICHARD WILLIAM /BD|THOMSON, RICHARD WILLIAM",cik,NaN,NaN
5,[1904.0],abraham bd|abraham|abraham securities corporation,"ABRAHAM & CO INC /BD|ABRAHAM & CO., INC.|ABRAHAM SECURITIES CORPORATION",cik,NaN,NaN
6,[1918.0],abrams allan edward|homeland securities financial services group|merchanthouse securities|wizer financial,"ABRAMS, ALLAN EDWARD|HOMELAND SECURITIES FINANCIAL SERVICES GROUP, INC.|THE MERCHANTHOUSE SECURITIES, INC.|WIZER FINANCIAL. INC.",cik,NaN,NaN
7,[1923.0],servidyne|abrams industries,"SERVIDYNE INC|ABRAMS INDUSTRIES INC|SERVIDYNE, INC.","compustat,cik",NaN,NaN
8,[1961.0],worlds|academic computer systems|worlds com,"WORLDS INC|ACADEMIC COMPUTER SYSTEMS INC|WORLDS COM INC|WORLDS.COM, INC.","compustat,cik",NaN,NaN
9,[2034.0],aceto,ACETO CORP,"compustat,cik",NaN,NaN


In [22]:
crosswalk_with_fdic[crosswalk_with_fdic['all_fdic_aliases'].notna()]

,cik,standardized_names,aliases,sources,std_name,all_fdic_aliases
1409,[73124.0],northern trust,NORTHERN TRUST CORP|The Northern Trust Company,"compustat,cik,FDIC",northern trust,The Northern Trust Company
2620,[316744.0],bessemer trust national association,"BESSEMER TRUST CO NATIONAL ASSOCIATION|BESSEMER TRUST COMPANY NATIONAL ASSOCIATION|Bessemer Trust Company, National Association","cik,FDIC",bessemer trust national association,"Bessemer Trust Company, National Association"
3338,[714395.0],german american bancorp,"GERMAN AMERICAN BANCORP INC|GERMAN AMERICAN BANCORP, INC.|GERMAN AMERICAN BANCORP|German American Bancorp","compustat,cik,FDIC",german american bancorp,German American Bancorp
3989,[740806.0],f m bank,F & M BANK CORP|F&M BANK CORP|F & M Bank|F&M Bank|F & M Bank|F&M Bank|F & M Bank|F&M Bank|F&M Bank|F & M Bank|F&M Bank|F&M Bank,"compustat,cik,FDIC",f m bank,F & M Bank|F&M Bank|F & M Bank|F&M Bank|F & M Bank|F&M Bank|F&M Bank|F & M Bank|F&M Bank|F&M Bank
4739,[778972.0],firstbank,FIRSTBANK CORP|Firstbank|FirstBank|FirstBank|FirstBank|FirstBank|FirstBank|Firstbank|FirstBank|FIRSTBANK,"compustat,cik,FDIC",firstbank,Firstbank|FirstBank|FirstBank|FirstBank|FirstBank|FirstBank|Firstbank|FirstBank|FIRSTBANK
5765,[810689.0],bank granite,BANK OF GRANITE CORP|Bank of Granite|Bank of Granite,"compustat,cik,FDIC",bank granite,Bank of Granite|Bank of Granite
12902,[1007273.0],bank south carolina,BANK SOUTH CAROLINA CORP|BANK OF SOUTH CAROLINA CORP|The Bank of South Carolina,"compustat,cik,FDIC",bank south carolina,The Bank of South Carolina
12976,[1008932.0],pnc bank national association,"PNC BANK NATIONAL ASSOCIATION/|PNC BANK, NATIONAL ASSOCIATION|PNC Bank, National Association|PNC Bank, National Association","cik,FDIC",pnc bank national association,"PNC Bank, National Association|PNC Bank, National Association"
15182,[1042729.0],mercantile bank,MERCANTILE BANK CORP|Mercantile Bank|Mercantile Bank|Mercantile Bank|Mercantile Bank|Mercantile Bank|Mercantile Bank|Mercantile Bank|Mercantile Bank,"compustat,cik,FDIC",mercantile bank,Mercantile Bank|Mercantile Bank|Mercantile Bank|Mercantile Bank|Mercantile Bank|Mercantile Bank|Mercantile Bank|Mercantile Bank
15940,[1053584.0],macatawa bank,MACATAWA BANK CORP|Macatawa Bank,"compustat,cik,FDIC",macatawa bank,Macatawa Bank


In [15]:
# clean up the crosswalk of FDIC + CIK + Compustat 

In [16]:
# need to isolate remaining data that couldn't be matched by CIK
# checks the size of each group by CIK
cik_group_sizes = all_names_df.groupby('cik')['cik'].transform('size')
processed_rows_mask = (cik_group_sizes > 1)
remaining_df = all_names_df[~processed_rows_mask].copy()

total_rows = len(all_names_df)
processed_rows_count = processed_rows_mask.sum()
remaining_rows_count = len(remaining_df)
print(f"Total rows: {total_rows}")
print(f"Processed rows (matched by CIK): {processed_rows_count}")
print(f"Remaining rows to process: {remaining_rows_count}")

Total rows: 915294
Processed rows (matched by CIK): 133396
Remaining rows to process: 781898


In [17]:
# Remaining rows to process will go through other matching methods including fuzzy matching and regex-based matching

grouped_by_std_name = remaining_df.groupby('std_name')
confident_matches_std_name = []

for name, group in grouped_by_std_name:
    # A "match" means this std_name appeared in more than one row
    if len(group) > 1:
        # Aggregate all unique keys and aliases
        keys = {
            'std_name': name,
            'aliases': '|'.join(group['raw_name'].dropna().unique()),
            'sources': ','.join(group['source'].unique()),
            'CIK': group['cik'].dropna().unique().tolist(),
        }
        confident_matches_std_name.append(keys)
pd.set_option('display.max_rows', None)
confident_crosswalk_std_name = pd.DataFrame(confident_matches_std_name)
print(f"  Found {len(confident_crosswalk_std_name)} additional clusters based on std_name.")
print(confident_crosswalk_std_name.head(100))
confident_crosswalk_std_name.head(10)

  Found 10949 additional clusters based on std_name.
                                                        std_name  \
0                                -depth partners global equities   
1                                        -operative bank concord   
2                                                    180 jamaica   
3                                                           180s   
4                                            1861 capital access   
5                                                       1st bank   
6                                                1st choice bank   
7                                             1st community bank   
8                                            1st enterprise bank   
9                                         1st financial bank usa   
10                                               1st source bank   
11                                                1st state bank   
12                                               1st united ban

,std_name,aliases,sources,CIK
0,-depth partners global equities,IN-DEPTH PARTNERS GLOBAL EQUITIES LLC|IN-DEPTH PARTNERS GLOBAL EQUITIES LTD.,cik,"[1882352.0, 1882346.0]"
1,-operative bank concord,The Co-operative Bank of Concord,fdic,[]
2,180 jamaica,180 JAMAICA CORP.|180 JAMAICA INC,cik,"[1294080.0, 1051663.0]"
3,180s,180S INC|180S LLC,cik,"[1305605.0, 1295552.0]"
4,1861 capital access,1861 CAPITAL ACCESS LLC|1861 CAPITAL ACCESS LTD,cik,"[1401997.0, 1404694.0]"
5,1st bank,1ST BANK|1st*Bank|1st Bank,fdic,[]
6,1st choice bank,1st Choice Bank,fdic,[]
7,1st community bank,1st Community Bank,fdic,[]
8,1st enterprise bank,1ST ENTERPRISE BANK|1st Enterprise Bank,"compustat,fdic",[]
9,1st financial bank usa,1st Financial Bank USA|1ST FINANCIAL BANK USA,"fdic,cik",[1541523.0]


In [18]:
cik_df[cik_df['company_name'] == 'ADAMS JOHN S']

,Unnamed: 0,company_name,cik,std_name
16903,16903,ADAMS JOHN S,1185427.0,adams john s
16904,16904,ADAMS JOHN S,1363163.0,adams john s


In [19]:
# from collections import Counter
# import pandas as pd
# from tqdm.auto import tqdm

# def get_match_candidate_score(frequency_dict, org_name, candidate_match_name):
#     if not isinstance(org_name, str):
#         org_name = ""
#     if not isinstance(candidate_match_name, str):
#         candidate_match_name = ""
    
#     if not org_name or not candidate_match_name:
#         return 0.0
    
#     org_tokens = org_name.split(' ')
    
#     # tokenize the candidate match
#     candidate_match_tokens = set(candidate_match_name.split(" "))

#     # Calculate the match score
#     total_inverse_frequency = 0
#     total_matching_inverse_frequency = 0
#     tokenized_name = org_tokens
#     for token in tokenized_name:
#         token_frequency = frequency_dict.get(token, 999999) # if token not found, give high frequency to ignore it
#         total_inverse_frequency += 1.0/token_frequency
#         if token in candidate_match_tokens:
#             total_matching_inverse_frequency += 1.0/token_frequency
#     match_score = total_matching_inverse_frequency / total_inverse_frequency

#     # added by James
#     weight = 1/len(org_name)
#     longest_common_substring = get_longest_common_substring(org_name, candidate_match_name, len(org_name), len(candidate_match_name))
#     match_score -= weight * len(candidate_match_name)/len(longest_common_substring) - weight

#     return match_score

# still_unmatched_df = remaining_df[~remaining_df['std_name'].isin(confident_crosswalk_std_name['std_name'])].copy().sample(frac=0.01, random_state=42)  # Sample 5% for testing
# print(f"Records remaining after std_name matching: {len(still_unmatched_df)}")
# all_tokens = []
# for name in still_unmatched_df['std_name']:
#     if isinstance(name, str) and name:
#         all_tokens.extend(name.lower().split(' '))

# token_frequency_dict = Counter(all_tokens)

# # --- 2. Create Candidate Match Dictionary (Reverse Index) ---
# candidate_match_dict = {}

# # Iterate over all unmatched records to build the index
# for idx, row in still_unmatched_df.iterrows():
#     # Store necessary data: (unique_id, std_name, original_raw_name, source)
#     candidate_tuple = (
#         f"{row['source']}-{idx}", 
#         row['std_name'].lower(), 
#         row['raw_name'], 
#         row['source']
#     )
    
#     for token in row['std_name'].lower().split(" "):
#         if token not in candidate_match_dict:
#             candidate_match_dict[token] = []
        
#         # Add candidate to the list for that token
#         if candidate_tuple not in candidate_match_dict[token]:
#              candidate_match_dict[token].append(candidate_tuple)


# # Set your desired threshold for a match
# MATCH_THRESHOLD = 0.95 
# high_confidence_matches = []

# # Loop through every company name in the DataFrame
# for idx, row in tqdm(still_unmatched_df.iterrows(), total=len(still_unmatched_df), desc="Entity Matching"):
#     org_name = row['std_name'].lower()
    
#     # --- A. Find candidates using top 2 rarest tokens (Optimization) ---
#     org_tokens = org_name.split(" ")
#     org_token_frequencies = sorted(
#         [(token, token_frequency_dict.get(token, 1)) for token in org_tokens],
#         key=lambda x: x[1] # Sort by frequency (rarest first)
#     )
    
#     candidate_set = set() 
    
#     # Use the two most unique tokens to fetch potential candidates from the index
#     for most_unique_token, _ in org_token_frequencies[:2]:
#         if most_unique_token in candidate_match_dict:
#             for candidate_tuple in candidate_match_dict[most_unique_token]:
#                 candidate_idx = candidate_tuple[0].split('-')[-1]
#                 # Exclude self-comparison
#                 if str(idx) != candidate_idx: 
#                     candidate_set.add(candidate_tuple)
    
#     # --- B. Score Candidates and Store Matches ---
#     for candidate_tuple in candidate_set:
#         unique_id, candidate_name, original_name_candidate, source_candidate = candidate_tuple
        
#         # Call the single-score function to get similarity
#         match_score = get_match_candidate_score(
#             token_frequency_dict, org_name, candidate_name)
        
#         # Filter for high-confidence matches
#         if match_score >= MATCH_THRESHOLD:
#             # Record the full details of the high-confidence match
#             high_confidence_matches.append({
#                 'Name_1_STD': org_name,
#                 'Name_1_RAW': row['raw_name'],
#                 'Name_1_Source': row['source'],
#                 'Name_2_STD': candidate_name,
#                 'Name_2_RAW': original_name_candidate,
#                 'Name_2_Source': source_candidate,
#                 'Similarity_Score': match_score,
#             })

# # --- 3. Final DataFrame ---
# match_candidates_df = pd.DataFrame(high_confidence_matches)
# print("Entity Matching Complete. High-Confidence Matches DataFrame:")
# print(match_candidates_df.head(20))




